# Optimization experiments


Eksperimen sebelumnya mencakup:
- Logistic Regression, LightGBM, dan XGBoost sebagai baseline.
- Feature engineering: flags, agregat, rasio, dan riwayat.
- Pencarian parameter awal.
- CatBoost dengan kategori native.
- Blending probabilitas OOF dengan bobot sama.

Hasil pengembangan:
- Biaya terendah: blend_four_v05, sebesar 536.3333 per 1.000 observasi.
- Blend XGBoost–LightGBM: 536.4583 per 1.000 observasi.
- F1 tertinggi: CatBoost raw, sebesar 0.5261.

Selisih biaya antarblend sangat kecil. Belum ada model final.

## Tujuan eksperimen ini

Menguji apakah tuning CatBoost dengan Optuna dapat memperbaiki hasil
dibandingkan CatBoost baseline dan kandidat sebelumnya.

Ketentuan:
- Target positif: default = 1.
- Cost function: 5 × FN + FP.
- Hanya menggunakan training set dengan 5-fold stratified CV.
- Preprocessing berada di dalam pipeline setiap fold.
- Threshold dipilih menggunakan probabilitas OOF training.
- Tidak menggunakan class weighting atau oversampling.
- Test set belum digunakan untuk evaluasi.
- Semua hasil merupakan development scores, bukan final test scores.

In [1]:
import hashlib
import json
from importlib.metadata import version
from pathlib import Path
from typing import Any

import numpy as np
import optuna
import pandas as pd

from catboost import CatBoostClassifier
from IPython.display import display
from sklearn.base import clone
from sklearn.pipeline import Pipeline

from src import config
from src.data_loader import load_and_split
from src.experiments import run_oof_experiment, save_experiment
from src.feature_engineering import CreditFeatureEngineer
from src.native_preprocess import NativeCategoricalPreprocessor


split = load_and_split()
X_train = split.X_train
y_train = split.y_train

# Tidak menginspeksi atau mengevaluasi bagian test.
print(f"Training rows: {len(X_train):,}")
print(f"Optuna version: {version('optuna')}")


def find_latest_blend_report() -> Path:
    """Mencari checkpoint blending lengkap yang paling baru."""
    root = config.EXPERIMENTS_DIR / "blend_oof_v05"

    if not root.is_dir():
        raise FileNotFoundError(
            "Laporan blend_oof_v05 belum ditemukan. "
            "Jalankan cell penyimpanan blending di notebook sebelumnya."
        )

    required_files = [
        config.EXPERIMENT_FILES["metadata"],
        config.EXPERIMENT_FILES["summary"],
        config.EXPERIMENT_FILES["oof_predictions"],
    ]

    candidates = [
        directory
        for directory in root.iterdir()
        if directory.is_dir()
        and all(
            (directory / filename).is_file()
            for filename in required_files
        )
    ]

    if not candidates:
        raise FileNotFoundError("Tidak ada checkpoint blending lengkap.")

    return max(candidates, key=lambda directory: directory.name)


checkpoint_dir = find_latest_blend_report()

checkpoint_metadata = json.loads(
    (
        checkpoint_dir / config.EXPERIMENT_FILES["metadata"]
    ).read_text(encoding="utf-8")
)

previous_summary = pd.read_csv(
    checkpoint_dir / config.EXPERIMENT_FILES["summary"]
)

previous_predictions = pd.read_csv(
    checkpoint_dir / config.EXPERIMENT_FILES["oof_predictions"],
    float_precision="round_trip",
)

dataset_sha256 = hashlib.sha256(
    config.DATA_PATH.read_bytes()
).hexdigest()

if checkpoint_metadata["dataset_sha256"] != dataset_sha256:
    raise ValueError("Dataset berubah dibandingkan checkpoint blending.")

if not np.array_equal(
    previous_predictions["source_row_index"].to_numpy(),
    X_train.index.to_numpy(),
):
    raise ValueError("Urutan atau anggota training set berubah.")

if not np.array_equal(
    previous_predictions["y_true"].to_numpy(),
    y_train.to_numpy(),
):
    raise ValueError("Label training berbeda dari checkpoint.")

alignment_columns = ["source_row_index", "fold", "y_true"]

reference_alignment = previous_predictions[
    alignment_columns
].reset_index(drop=True)

print(f"Checkpoint: {checkpoint_dir}")

d:\proj-1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training rows: 24,000
Optuna version: 5.0.0
Checkpoint: D:\proj-1\reports\experiments\blend_oof_v05\20261005T092357545754Z


In [2]:
CAT_BASE_PARAMS: dict[str, Any] = dict(
    checkpoint_metadata["catboost_parameters"]
)

# JSON menyimpan tuple sebagai list. Kembalikan agar clone tetap aman.
CAT_BASE_PARAMS["cat_features"] = tuple(
    config.CATEGORICAL_COLUMNS
)

FEATURE_SETS: dict[str, tuple[str, ...]] = {
    "raw": (),
    "aggregate": tuple(config.FE_AGGREGATE_COLUMNS),
}

SEARCH_SPACE: dict[str, Any] = {
    "feature_set": ["raw", "aggregate"],
    "iterations": [400, 800, 1200],
    "depth": [3, 7],
    "learning_rate": [0.02, 0.10],
    "l2_leaf_reg": [1.0, 20.0],
}


def make_cat_pipeline(
    parameters: dict[str, Any],
) -> Pipeline:
    """Membangun pipeline CatBoost dari parameter satu trial."""
    model_parameters = dict(parameters)
    feature_set = str(model_parameters.pop("feature_set"))

    if feature_set not in FEATURE_SETS:
        raise ValueError(f"Feature set tidak dikenal: {feature_set}")

    selected_features = FEATURE_SETS[feature_set]
    steps: list[tuple[str, Any]] = []

    if selected_features:
        steps.append(
            (
                "feature_engineering",
                CreditFeatureEngineer(features=selected_features),
            )
        )

    steps.extend(
        [
            ("preprocess", NativeCategoricalPreprocessor()),
            (
                "model",
                CatBoostClassifier(
                    **(CAT_BASE_PARAMS | model_parameters)
                ),
            ),
        ]
    )

    return Pipeline(steps)


# Memeriksa masalah clone sebelum menjalankan pencarian panjang.
clone(make_cat_pipeline({"feature_set": "raw"}))

print("Pipeline dapat di-clone.")

Pipeline dapat di-clone.


In [3]:
source_modules = (
    "config",
    "data_loader",
    "preprocess",
    "feature_engineering",
    "native_preprocess",
    "experiments",
)

source_hashes = {
    name: hashlib.sha256(
        (
            config.PROJECT_ROOT / "src" / f"{name}.py"
        ).read_bytes()
    ).hexdigest()
    for name in source_modules
}

study_spec: dict[str, Any] = {
    "scope": "train_oof_catboost_optimization",
    "dataset_sha256": dataset_sha256,
    "cv_folds": config.CV_FOLDS,
    "cv_random_state": config.CV_RANDOM_STATE,
    "false_negative_cost": config.FALSE_NEGATIVE_COST,
    "false_positive_cost": config.FALSE_POSITIVE_COST,
    "base_parameters": CAT_BASE_PARAMS,
    "feature_sets": FEATURE_SETS,
    "search_space": SEARCH_SPACE,
    "sampler": {
        "name": "TPE",
        "seed": config.RANDOM_STATE,
        "n_startup_trials": 8,
    },
    "package_versions": {
        name: version(name)
        for name in (
            "optuna",
            "catboost",
            "scikit-learn",
            "numpy",
            "pandas",
        )
    },
    "source_hashes": source_hashes,
}

spec_json = json.dumps(
    study_spec,
    sort_keys=True,
    allow_nan=False,
)

study_signature = hashlib.sha256(
    spec_json.encode("utf-8")
).hexdigest()[:16]

study_dir = (
    config.EXPERIMENTS_DIR
    / "cat_optuna_v06"
    / study_signature
)
study_dir.mkdir(parents=True, exist_ok=True)

(
    study_dir / config.EXPERIMENT_FILES["metadata"]
).write_text(
    json.dumps(study_spec, indent=2, allow_nan=False),
    encoding="utf-8",
)

database_path = study_dir / "study.sqlite3"
storage_url = f"sqlite:///{database_path.resolve().as_posix()}"

study = optuna.create_study(
    study_name="catboost_v06",
    direction="minimize",
    sampler=optuna.samplers.TPESampler(
        seed=config.RANDOM_STATE,
        n_startup_trials=8,
    ),
    pruner=optuna.pruners.NopPruner(),
    storage=storage_url,
    load_if_exists=True,
)

# Dua konfigurasi baseline menjadi titik awal pencarian.
if not study.trials:
    for feature_set in FEATURE_SETS:
        study.enqueue_trial(
            {
                "feature_set": feature_set,
                "iterations": CAT_BASE_PARAMS["iterations"],
                "depth": CAT_BASE_PARAMS["depth"],
                "learning_rate": CAT_BASE_PARAMS["learning_rate"],
                "l2_leaf_reg": CAT_BASE_PARAMS["l2_leaf_reg"],
            }
        )

print(f"Study directory: {study_dir}")

[I 2026-10-05 16:46:23,065] A new study created in RDB with name: catboost_v06


Study directory: D:\proj-1\reports\experiments\cat_optuna_v06\36ef05f4e5af6381


In [6]:
CAT_BASE_PARAMS["task_type"] = "CPU"
CAT_BASE_PARAMS.pop("devices", None)

'0'

In [8]:
import hashlib
import json
from pathlib import Path
from typing import Any

import numpy as np
import optuna
import pandas as pd

from catboost import CatBoostClassifier
from IPython.display import display
from sklearn.base import clone
from sklearn.pipeline import Pipeline

from src import config
from src.data_loader import load_and_split
from src.experiments import run_oof_experiment, save_experiment
from src.feature_engineering import CreditFeatureEngineer
from src.native_preprocess import NativeCategoricalPreprocessor


# ============================================================
# 1. Memuat study CPU asli dari log eksperimenmu
# ============================================================

STUDY_SIGNATURE = "36ef05f4e5af6381"
TARGET_COMPLETED_TRIALS = 10

study_dir = (
    config.EXPERIMENTS_DIR
    / "cat_optuna_v06"
    / STUDY_SIGNATURE
)

database_path = study_dir / "study.sqlite3"
metadata_path = (
    study_dir / config.EXPERIMENT_FILES["metadata"]
)

if not database_path.is_file() or not metadata_path.is_file():
    raise FileNotFoundError(
        f"Study CPU tidak ditemukan: {study_dir}"
    )

study_spec: dict[str, Any] = json.loads(
    metadata_path.read_text(encoding="utf-8")
)

dataset_sha256 = hashlib.sha256(
    config.DATA_PATH.read_bytes()
).hexdigest()

if dataset_sha256 != study_spec["dataset_sha256"]:
    raise ValueError(
        "Dataset berubah dibandingkan study CPU sebelumnya."
    )

# Memastikan kode yang digunakan masih sama.
for module_name, expected_hash in study_spec["source_hashes"].items():
    module_path = (
        config.PROJECT_ROOT / "src" / f"{module_name}.py"
    )

    current_hash = hashlib.sha256(
        module_path.read_bytes()
    ).hexdigest()

    if current_hash != expected_hash:
        raise ValueError(
            f"Kode {module_name}.py berubah. "
            "Study lama tidak dilanjutkan agar hasil tidak tercampur."
        )


# ============================================================
# 2. Memuat training set dan konfigurasi tersimpan
# ============================================================

split = load_and_split()
X_train = split.X_train
y_train = split.y_train

# Bagian test tidak diinspeksi atau dievaluasi.
print(f"Training rows: {len(X_train):,}")

CAT_BASE_PARAMS: dict[str, Any] = dict(
    study_spec["base_parameters"]
)

if CAT_BASE_PARAMS["task_type"] != "CPU":
    raise ValueError("Checkpoint yang dimuat bukan study CPU.")

# Mengembalikan list dari JSON menjadi tuple agar clone aman.
CAT_BASE_PARAMS["cat_features"] = tuple(
    CAT_BASE_PARAMS["cat_features"]
)

FEATURE_SETS: dict[str, tuple[str, ...]] = {
    name: tuple(columns)
    for name, columns in study_spec["feature_sets"].items()
}

SEARCH_SPACE: dict[str, Any] = study_spec["search_space"]


# ============================================================
# 3. Membangun pipeline untuk setiap trial
# ============================================================

def make_cat_pipeline(
    parameters: dict[str, Any],
) -> Pipeline:
    """Membangun pipeline CatBoost CPU untuk satu konfigurasi."""
    model_parameters = dict(parameters)
    feature_set = str(model_parameters.pop("feature_set"))

    if feature_set not in FEATURE_SETS:
        raise ValueError(
            f"Feature set tidak dikenal: {feature_set}"
        )

    selected_features = FEATURE_SETS[feature_set]
    steps: list[tuple[str, Any]] = []

    if selected_features:
        steps.append(
            (
                "feature_engineering",
                CreditFeatureEngineer(
                    features=selected_features,
                ),
            )
        )

    steps.extend(
        [
            ("preprocess", NativeCategoricalPreprocessor()),
            (
                "model",
                CatBoostClassifier(
                    **(CAT_BASE_PARAMS | model_parameters)
                ),
            ),
        ]
    )

    return Pipeline(steps)


# Memeriksa cloning sebelum menjalankan training.
clone(make_cat_pipeline({"feature_set": "raw"}))


# ============================================================
# 4. Mendefinisikan ulang objective yang sebelumnya dihapus
# ============================================================

def objective(trial: optuna.Trial) -> float:
    """Meminimalkan biaya OOF dan menyimpan laporan setiap trial."""
    parameters: dict[str, Any] = {
        "feature_set": trial.suggest_categorical(
            "feature_set",
            SEARCH_SPACE["feature_set"],
        ),
        "iterations": trial.suggest_categorical(
            "iterations",
            SEARCH_SPACE["iterations"],
        ),
        "depth": trial.suggest_int(
            "depth",
            *SEARCH_SPACE["depth"],
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate",
            *SEARCH_SPACE["learning_rate"],
            log=True,
        ),
        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg",
            *SEARCH_SPACE["l2_leaf_reg"],
            log=True,
        ),
    }

    pipeline = make_cat_pipeline(parameters)

    experiment_id = (
        f"cat_optuna_v06_{STUDY_SIGNATURE}_{trial.number:03d}"
    )

    print(f"Menjalankan {experiment_id}...", flush=True)

    result = run_oof_experiment(
        pipeline=pipeline,
        X=X_train,
        y=y_train,
    )

    # Memeriksa kesesuaian baris hasil OOF.
    if not np.array_equal(
        result.predictions["source_row_index"].to_numpy(),
        X_train.index.to_numpy(),
    ):
        raise ValueError("Urutan baris prediksi OOF berubah.")

    if not np.array_equal(
        result.predictions["y_true"].to_numpy(),
        y_train.to_numpy(),
    ):
        raise ValueError("Label prediksi OOF tidak sesuai.")

    run_dir = save_experiment(
        result=result,
        pipeline=pipeline,
        experiment_id=experiment_id,
        numeric_scaling="none",
    )

    summary = {
        key: value.item()
        if isinstance(value, np.generic)
        else value
        for key, value in result.summary.items()
    }

    trial.set_user_attr("experiment_id", experiment_id)
    trial.set_user_attr("report_dir", str(run_dir))
    trial.set_user_attr("summary", summary)

    cost = float(summary["oof_cost_per_1000"])
    f1 = float(summary["oof_f1_at_selected_threshold"])

    print(
        f"Biaya/1.000={cost:.4f}; F1={f1:.4f}",
        flush=True,
    )

    return cost


# ============================================================
# 5. Memuat study dan melanjutkan trial yang belum selesai
# ============================================================

storage_url = f"sqlite:///{database_path.resolve().as_posix()}"

study = optuna.load_study(
    study_name="catboost_v06",
    storage=storage_url,
    sampler=optuna.samplers.TPESampler(
        seed=study_spec["sampler"]["seed"],
        n_startup_trials=study_spec["sampler"]["n_startup_trials"],
    ),
    pruner=optuna.pruners.NopPruner(),
)

completed_trials = study.get_trials(
    states=(optuna.trial.TrialState.COMPLETE,),
)

remaining_trials = max(
    0,
    TARGET_COMPLETED_TRIALS - len(completed_trials),
)

print(f"Perangkat: {CAT_BASE_PARAMS['task_type']}")
print(f"Trial selesai: {len(completed_trials)}")
print(f"Trial tambahan: {remaining_trials}")

if remaining_trials:
    study.optimize(
        objective,
        n_trials=remaining_trials,
        n_jobs=1,
    )
else:
    print("Target trial selesai sudah tercapai.")


# ============================================================
# 6. Menyimpan dan menampilkan ringkasan
# ============================================================

completed_trials = study.get_trials(
    states=(optuna.trial.TrialState.COMPLETE,),
)

if not completed_trials:
    raise RuntimeError("Belum ada trial yang berhasil selesai.")

rows: list[dict[str, Any]] = []

for trial in completed_trials:
    rows.append(
        {
            "experiment_id": trial.user_attrs["experiment_id"],
            "trial_number": trial.number,
            "feature_set": trial.params["feature_set"],
            **trial.user_attrs["summary"],
            "parameters": json.dumps(
                trial.params,
                sort_keys=True,
            ),
            "report_dir": trial.user_attrs["report_dir"],
        }
    )

cat_optuna_search = pd.DataFrame(rows).sort_values(
    ["oof_cost_per_1000", "trial_number"],
    ignore_index=True,
)

cat_optuna_search.to_csv(
    study_dir / config.EXPERIMENT_FILES["summary"],
    index=False,
)

display_columns = [
    "experiment_id",
    "feature_set",
    "oof_cost_per_1000",
    "oof_f1_at_selected_threshold",
    "oof_roc_auc",
    "oof_average_precision",
    "fn",
    "fp",
    "threshold",
]

display(
    cat_optuna_search[display_columns]
    .head(5)
    .round(4)
)

best_trial = study.best_trial

# Template pipeline; belum dilatih pada seluruh training set.
best_cat_pipeline = make_cat_pipeline(best_trial.params)

best_cat_report_dir = Path(
    best_trial.user_attrs["report_dir"]
)

print("Parameter terbaik:")
print(json.dumps(best_trial.params, indent=2))

print(f"Biaya terbaik/1.000: {best_trial.value:.4f}")
print(f"Ringkasan tersimpan: {study_dir}")

Training rows: 24,000
Perangkat: CPU
Trial selesai: 3
Trial tambahan: 7
Menjalankan cat_optuna_v06_36ef05f4e5af6381_005...
Biaya/1.000=544.5000; F1=0.5173


[I 2026-10-05 17:41:06,895] Trial 5 finished with value: 544.5 and parameters: {'feature_set': 'aggregate', 'iterations': 400, 'depth': 3, 'learning_rate': 0.021959816073193316, 'l2_leaf_reg': 13.39433470675048}. Best is trial 1 with value: 542.7083333333334.


Menjalankan cat_optuna_v06_36ef05f4e5af6381_006...
Biaya/1.000=542.3750; F1=0.5236


[I 2026-10-05 17:49:17,361] Trial 6 finished with value: 542.375 and parameters: {'feature_set': 'aggregate', 'iterations': 800, 'depth': 4, 'learning_rate': 0.02679909904436604, 'l2_leaf_reg': 1.7322667470546256}. Best is trial 6 with value: 542.375.


Menjalankan cat_optuna_v06_36ef05f4e5af6381_007...
Biaya/1.000=542.3750; F1=0.5226


...

[I 2026-10-05 18:09:49,471] Trial 10 finished with value: 541.2916666666666 and parameters: {'feature_set': 'aggregate', 'iterations': 800, 'depth': 5, 'learning_rate': 0.031706957479144204, 'l2_leaf_reg': 1.5369998974773227}. Best is trial 10 with value: 541.2916666666666.


Menjalankan cat_optuna_v06_36ef05f4e5af6381_011...
Biaya/1.000=539.3333; F1=0.4965


[I 2026-10-05 18:13:46,643] Trial 11 finished with value: 539.3333333333334 and parameters: {'feature_set': 'aggregate', 'iterations': 800, 'depth': 6, 'learning_rate': 0.06074240675387992, 'l2_leaf_reg': 15.244157553717777}. Best is trial 11 with value: 539.3333333333334.


,experiment_id,feature_set,oof_cost_per_1000,oof_f1_at_selected_threshold,oof_roc_auc,oof_average_precision,fn,fp,threshold
0,cat_optuna_v06_36ef05f4e5af6381_011,aggregate,539.3333,0.4965,0.7850,0.5571,1104,7424,0.1437
1,cat_optuna_v06_36ef05f4e5af6381_010,aggregate,541.2917,0.5212,0.7862,0.5590,1496,5511,0.1725
2,cat_optuna_v06_36ef05f4e5af6381_006,aggregate,542.3750,0.5236,0.7855,0.5603,1539,5322,0.1756
3,cat_optuna_v06_36ef05f4e5af6381_007,aggregate,542.3750,0.5226,0.7855,0.5599,1527,5382,0.1742
4,cat_optuna_v06_36ef05f4e5af6381_009,aggregate,542.3750,0.5234,0.7842,0.5569,1537,5332,0.1742


Parameter terbaik:
{
  "feature_set": "aggregate",
  "iterations": 800,
  "depth": 6,
  "learning_rate": 0.06074240675387992,
  "l2_leaf_reg": 15.244157553717777
}
Biaya terbaik/1.000: 539.3333
Ringkasan tersimpan: D:\proj-1\reports\experiments\cat_optuna_v06\36ef05f4e5af6381


## CatBoost optimization checkpoint

CatBoost CPU telah mencapai checkpoint pencarian awal.

Konfigurasi biaya terendah:
- Feature set: aggregate
- Iterations: 800
- Depth: 6
- Learning rate: 0.06074240675387992
- L2 leaf regularization: 15.244157553717777

Hasil development OOF:
- Cost per 1,000: 539.3333
- F1 at cost-selected threshold: 0.4965
- ROC-AUC: 0.7850
- Average precision: 0.5571
- FN: 1104
- FP: 7424

Biaya turun sekitar 0.622% dibandingkan CatBoost aggregate baseline.
ROC-AUC dan AP tidak meningkat. Kandidat disimpan untuk eksperimen
ensemble; belum menjadi model final.

Eksperimen berikutnya: Optuna XGBoost, menggunakan konfigurasi terbaik
dari pencarian sebelumnya sebagai titik awal.

In [9]:
import hashlib
import json
from importlib.metadata import version
from pathlib import Path
from time import perf_counter
from typing import Any

import numpy as np
import optuna
import pandas as pd

from IPython.display import display
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

from src import config
from src.data_loader import load_and_split
from src.experiments import run_oof_experiment, save_experiment
from src.feature_engineering import CreditFeatureEngineer
from src.preprocess import build_preprocessor


# Nama variabel XGBoost dipisahkan agar konfigurasi CatBoost tetap tersedia.
xgb_split = load_and_split()
xgb_X_train = xgb_split.X_train
xgb_y_train = xgb_split.y_train

xgb_dataset_sha256 = hashlib.sha256(
    config.DATA_PATH.read_bytes()
).hexdigest()


def find_latest_xgb_search() -> Path:
    """Mencari ringkasan pencarian XGBoost sebelumnya."""
    root = (
        config.EXPERIMENTS_DIR
        / "xgb_aggregate_search_v03"
    )

    if not root.is_dir():
        raise FileNotFoundError(
            f"Laporan pencarian XGBoost tidak ditemukan: {root}"
        )

    candidates = [
        directory
        for directory in root.iterdir()
        if directory.is_dir()
        and (
            directory / config.EXPERIMENT_FILES["summary"]
        ).is_file()
    ]

    if not candidates:
        raise FileNotFoundError(
            "Tidak ada ringkasan pencarian XGBoost lengkap."
        )

    return max(candidates, key=lambda directory: directory.name)


xgb_previous_search_dir = find_latest_xgb_search()

xgb_previous_search = pd.read_csv(
    xgb_previous_search_dir
    / config.EXPERIMENT_FILES["summary"]
).sort_values(
    "oof_cost_per_1000",
    ignore_index=True,
)

xgb_previous_best = xgb_previous_search.iloc[0]

xgb_previous_report_dir = Path(
    str(xgb_previous_best["report_dir"])
)

xgb_previous_metadata = json.loads(
    (
        xgb_previous_report_dir
        / config.EXPERIMENT_FILES["metadata"]
    ).read_text(encoding="utf-8")
)

if (
    xgb_previous_metadata["dataset_sha256"]
    != xgb_dataset_sha256
):
    raise ValueError("Dataset berbeda dari eksperimen XGBoost lama.")

if (
    xgb_previous_metadata["cv_folds"] != config.CV_FOLDS
    or xgb_previous_metadata["cv_random_state"]
    != config.CV_RANDOM_STATE
):
    raise ValueError("Konfigurasi fold berbeda dari eksperimen lama.")

xgb_previous_predictions = pd.read_csv(
    xgb_previous_report_dir
    / config.EXPERIMENT_FILES["oof_predictions"],
    float_precision="round_trip",
)

XGB_ALIGNMENT_COLUMNS = ["source_row_index", "fold", "y_true"]

xgb_reference_alignment = xgb_previous_predictions[
    XGB_ALIGNMENT_COLUMNS
].reset_index(drop=True)

if not np.array_equal(
    xgb_reference_alignment["source_row_index"].to_numpy(),
    xgb_X_train.index.to_numpy(),
):
    raise ValueError("Anggota atau urutan training set berubah.")

if not np.array_equal(
    xgb_reference_alignment["y_true"].to_numpy(),
    xgb_y_train.to_numpy(),
):
    raise ValueError("Label training berbeda dari eksperimen lama.")


# Parameter eksplisit yang tersimpan dalam tabel pencarian lama.
xgb_previous_best_parameters = json.loads(
    str(xgb_previous_best["parameters"])
)

XGB_FIXED_PARAMS: dict[str, Any] = {
    **config.XGB_BASELINE_PARAMS,
    **xgb_previous_best_parameters,
    "device": "cpu",
    "tree_method": "hist",
    "n_jobs": 4,
    "verbosity": 0,
}

XGB_FEATURES = tuple(config.FE_AGGREGATE_COLUMNS)

XGB_SEARCH_SPACE: dict[str, Any] = {
    "n_estimators": [200, 400, 800, 1200],
    "max_depth": [2, 6],
    "learning_rate": [0.015, 0.10],
    "min_child_weight": [1.0, 20.0],
    "reg_lambda": [0.5, 30.0],
    "reg_alpha": [0.0, 5.0],
    "subsample": [0.7, 1.0],
    "colsample_bytree": [0.7, 1.0],
}


def make_xgb_pipeline(
    parameters: dict[str, Any],
) -> Pipeline:
    """Membangun pipeline XGBoost dengan fitur agregat."""
    preprocessor = build_preprocessor(
        numeric_scaling="none",
        additional_numeric_columns=XGB_FEATURES,
    ).set_output(transform="pandas")

    return Pipeline(
        [
            (
                "feature_engineering",
                CreditFeatureEngineer(features=XGB_FEATURES),
            ),
            ("preprocess", preprocessor),
            (
                "model",
                XGBClassifier(
                    **(XGB_FIXED_PARAMS | parameters)
                ),
            ),
        ]
    )


clone(make_xgb_pipeline({}))

print(f"Training rows: {len(xgb_X_train):,}")
print(
    "Biaya XGBoost sebelumnya/1.000: "
    f"{xgb_previous_best['oof_cost_per_1000']:.4f}"
)
print("Pipeline XGBoost siap.")

Training rows: 24,000
Biaya XGBoost sebelumnya/1.000: 537.7083
Pipeline XGBoost siap.


In [10]:
# ============================================================
# Identitas dan penyimpanan study
# ============================================================

xgb_source_modules = (
    "config",
    "data_loader",
    "preprocess",
    "feature_engineering",
    "experiments",
)

xgb_source_hashes = {
    name: hashlib.sha256(
        (
            config.PROJECT_ROOT / "src" / f"{name}.py"
        ).read_bytes()
    ).hexdigest()
    for name in xgb_source_modules
}

# Dua kandidat biaya terendah menjadi titik awal.
xgb_anchor_parameters: list[dict[str, Any]] = []

for parameter_json in xgb_previous_search.head(2)["parameters"]:
    previous_parameters = json.loads(str(parameter_json))

    anchor = {
        name: previous_parameters.get(
            name,
            XGB_FIXED_PARAMS.get(name, 0.0),
        )
        for name in XGB_SEARCH_SPACE
    }

    if anchor not in xgb_anchor_parameters:
        xgb_anchor_parameters.append(anchor)

xgb_study_spec: dict[str, Any] = {
    "scope": "train_oof_xgboost_optimization",
    "dataset_sha256": xgb_dataset_sha256,
    "cv_folds": config.CV_FOLDS,
    "cv_random_state": config.CV_RANDOM_STATE,
    "false_negative_cost": config.FALSE_NEGATIVE_COST,
    "false_positive_cost": config.FALSE_POSITIVE_COST,
    "base_parameters": XGB_FIXED_PARAMS,
    "engineered_features": XGB_FEATURES,
    "search_space": XGB_SEARCH_SPACE,
    "anchors": xgb_anchor_parameters,
    "sampler_seed": config.RANDOM_STATE,
    "n_startup_trials": 8,
    "source_hashes": xgb_source_hashes,
    "package_versions": {
        name: version(name)
        for name in (
            "xgboost",
            "optuna",
            "scikit-learn",
            "numpy",
            "pandas",
        )
    },
}

xgb_spec_json = json.dumps(
    xgb_study_spec,
    sort_keys=True,
    allow_nan=False,
)

xgb_study_signature = hashlib.sha256(
    xgb_spec_json.encode("utf-8")
).hexdigest()[:16]

xgb_study_dir = (
    config.EXPERIMENTS_DIR
    / "xgb_optuna_v07"
    / xgb_study_signature
)
xgb_study_dir.mkdir(parents=True, exist_ok=True)

(
    xgb_study_dir / config.EXPERIMENT_FILES["metadata"]
).write_text(
    json.dumps(xgb_study_spec, indent=2, allow_nan=False),
    encoding="utf-8",
)

xgb_database_path = xgb_study_dir / "study.sqlite3"

xgb_study = optuna.create_study(
    study_name="xgboost_v07",
    direction="minimize",
    sampler=optuna.samplers.TPESampler(
        seed=config.RANDOM_STATE,
        n_startup_trials=8,
    ),
    pruner=optuna.pruners.NopPruner(),
    storage=f"sqlite:///{xgb_database_path.resolve().as_posix()}",
    load_if_exists=True,
)

if not xgb_study.trials:
    for anchor in xgb_anchor_parameters:
        xgb_study.enqueue_trial(anchor)


# ============================================================
# Objective
# ============================================================

def xgb_objective(trial: optuna.Trial) -> float:
    """Mencari parameter yang meminimalkan biaya prediksi OOF."""
    parameters: dict[str, Any] = {
        "n_estimators": trial.suggest_categorical(
            "n_estimators",
            XGB_SEARCH_SPACE["n_estimators"],
        ),
        "max_depth": trial.suggest_int(
            "max_depth",
            *XGB_SEARCH_SPACE["max_depth"],
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate",
            *XGB_SEARCH_SPACE["learning_rate"],
            log=True,
        ),
        "min_child_weight": trial.suggest_float(
            "min_child_weight",
            *XGB_SEARCH_SPACE["min_child_weight"],
            log=True,
        ),
        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            *XGB_SEARCH_SPACE["reg_lambda"],
            log=True,
        ),
        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            *XGB_SEARCH_SPACE["reg_alpha"],
        ),
        "subsample": trial.suggest_float(
            "subsample",
            *XGB_SEARCH_SPACE["subsample"],
        ),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            *XGB_SEARCH_SPACE["colsample_bytree"],
        ),
    }

    experiment_id = (
        f"xgb_optuna_v07_{xgb_study_signature}_{trial.number:03d}"
    )

    print(f"Menjalankan {experiment_id}...", flush=True)
    started = perf_counter()

    pipeline = make_xgb_pipeline(parameters)

    result = run_oof_experiment(
        pipeline=pipeline,
        X=xgb_X_train,
        y=xgb_y_train,
    )

    current_alignment = result.predictions[
        XGB_ALIGNMENT_COLUMNS
    ].reset_index(drop=True)

    if not xgb_reference_alignment.equals(current_alignment):
        raise ValueError(
            "Baris, label, atau fold berbeda dari eksperimen lama."
        )

    run_dir = save_experiment(
        result=result,
        pipeline=pipeline,
        experiment_id=experiment_id,
        numeric_scaling="none",
    )

    summary = {
        key: value.item()
        if isinstance(value, np.generic)
        else value
        for key, value in result.summary.items()
    }

    elapsed = perf_counter() - started

    trial.set_user_attr("experiment_id", experiment_id)
    trial.set_user_attr("report_dir", str(run_dir))
    trial.set_user_attr("summary", summary)
    trial.set_user_attr("seconds", elapsed)

    cost = float(summary["oof_cost_per_1000"])

    print(
        f"Biaya/1.000={cost:.4f}; "
        f"F1={summary['oof_f1_at_selected_threshold']:.4f}; "
        f"durasi={elapsed:.1f} detik",
        flush=True,
    )

    return cost


# ============================================================
# Jalankan pencarian
# ============================================================

TARGET_XGB_TRIALS = 30
XGB_TIMEOUT_SECONDS = 3600

xgb_completed = xgb_study.get_trials(
    states=(optuna.trial.TrialState.COMPLETE,),
)

xgb_remaining = max(
    0,
    TARGET_XGB_TRIALS - len(xgb_completed),
)

print(f"Trial selesai: {len(xgb_completed)}")
print(f"Trial tambahan maksimal: {xgb_remaining}")

if xgb_remaining:
    xgb_study.optimize(
        xgb_objective,
        n_trials=xgb_remaining,
        timeout=XGB_TIMEOUT_SECONDS,
        n_jobs=1,
    )


# ============================================================
# Ringkasan
# ============================================================

xgb_completed = xgb_study.get_trials(
    states=(optuna.trial.TrialState.COMPLETE,),
)

if not xgb_completed:
    raise RuntimeError("Belum ada trial XGBoost yang selesai.")

xgb_rows: list[dict[str, Any]] = []

for trial in xgb_completed:
    xgb_rows.append(
        {
            "experiment_id": trial.user_attrs["experiment_id"],
            "trial_number": trial.number,
            **trial.user_attrs["summary"],
            "seconds": trial.user_attrs["seconds"],
            "parameters": json.dumps(
                trial.params,
                sort_keys=True,
            ),
            "report_dir": trial.user_attrs["report_dir"],
        }
    )

xgb_optuna_search = pd.DataFrame(xgb_rows).sort_values(
    ["oof_cost_per_1000", "trial_number"],
    ignore_index=True,
)

xgb_optuna_search.to_csv(
    xgb_study_dir / config.EXPERIMENT_FILES["summary"],
    index=False,
)

xgb_display_columns = [
    "experiment_id",
    "oof_cost_per_1000",
    "oof_f1_at_selected_threshold",
    "oof_roc_auc",
    "oof_average_precision",
    "fn",
    "fp",
    "threshold",
    "seconds",
]

display(
    xgb_optuna_search[xgb_display_columns]
    .head(5)
    .round(4)
)

xgb_best_trial = xgb_study.best_trial

# Template; belum dilatih pada seluruh training set.
best_xgb_optuna_pipeline = make_xgb_pipeline(
    xgb_best_trial.params
)

best_xgb_optuna_report_dir = Path(
    xgb_best_trial.user_attrs["report_dir"]
)

best_xgb_optuna_oof = pd.read_csv(
    best_xgb_optuna_report_dir
    / config.EXPERIMENT_FILES["oof_predictions"],
    float_precision="round_trip",
)

xgb_old_cost = float(
    xgb_previous_best["oof_cost_per_1000"]
)

xgb_new_cost = float(xgb_best_trial.value)

xgb_reduction_pct = (
    (xgb_old_cost - xgb_new_cost) / xgb_old_cost * 100
)

print("Parameter terbaik:")
print(json.dumps(xgb_best_trial.params, indent=2))

print(f"Biaya lama/1.000: {xgb_old_cost:.4f}")
print(f"Biaya baru/1.000: {xgb_new_cost:.4f}")
print(f"Penurunan biaya relatif: {xgb_reduction_pct:+.4f}%")
print(f"Trial selesai: {len(xgb_completed)}")
print(f"Ringkasan tersimpan: {xgb_study_dir}")

[I 2026-10-05 18:22:17,314] A new study created in RDB with name: xgboost_v07


Trial selesai: 0
Trial tambahan maksimal: 30
Menjalankan xgb_optuna_v07_9d97e38c114bd0a2_000...
Biaya/1.000=537.7083; F1=0.5052; durasi=16.8 detik


[I 2026-10-05 18:22:34,325] Trial 0 finished with value: 537.7083333333334 and parameters: {'n_estimators': 800, 'max_depth': 3, 'learning_rate': 0.05, 'min_child_weight': 1.0, 'reg_lambda': 5.0, 'reg_alpha': 0.0, 'subsample': 0.8, 'colsample_bytree': 1.0}. Best is trial 0 with value: 537.7083333333334.


Menjalankan xgb_optuna_v07_9d97e38c114bd0a2_001...
Biaya/1.000=537.8750; F1=0.5239; durasi=13.1 detik


[I 2026-10-05 18:22:47,736] Trial 1 finished with value: 537.875 and parameters: {'n_estimators': 400, 'max_depth': 5, 'learning_rate': 0.03, 'min_child_weight': 10.0, 'reg_lambda': 15.0, 'reg_alpha': 0.0, 'subsample': 0.8, 'colsample_bytree': 1.0}. Best is trial 0 with value: 537.7083333333334.


...

[I 2026-10-05 18:26:51,419] Trial 28 finished with value: 536.9583333333334 and parameters: {'n_estimators': 1200, 'max_depth': 3, 'learning_rate': 0.02452133330013035, 'min_child_weight': 5.117069566547082, 'reg_lambda': 5.626381322433853, 'reg_alpha': 3.868756193423694, 'subsample': 0.7338050416668733, 'colsample_bytree': 0.7126115528467195}. Best is trial 16 with value: 535.625.


Menjalankan xgb_optuna_v07_9d97e38c114bd0a2_029...
Biaya/1.000=536.5000; F1=0.5142; durasi=10.6 detik


[I 2026-10-05 18:27:02,062] Trial 29 finished with value: 536.5 and parameters: {'n_estimators': 1200, 'max_depth': 3, 'learning_rate': 0.027501801885342656, 'min_child_weight': 1.6489119814767488, 'reg_lambda': 1.3753648818692936, 'reg_alpha': 3.1102403845062114, 'subsample': 0.7800924828500597, 'colsample_bytree': 0.7521963192507622}. Best is trial 16 with value: 535.625.


,experiment_id,oof_cost_per_1000,oof_f1_at_selected_threshold,oof_roc_auc,oof_average_precision,fn,fp,threshold,seconds
0,xgb_optuna_v07_9d97e38c114bd0a2_016,535.6250,0.5115,0.7864,0.5562,1299,6360,0.1620,7.9929
1,xgb_optuna_v07_9d97e38c114bd0a2_017,535.7500,0.5105,0.7867,0.5579,1285,6433,0.1613,9.3228
2,xgb_optuna_v07_9d97e38c114bd0a2_022,536.1667,0.5146,0.7870,0.5594,1350,6118,0.1662,9.1461
3,xgb_optuna_v07_9d97e38c114bd0a2_029,536.5000,0.5142,0.7860,0.5574,1348,6136,0.1669,10.5684
4,xgb_optuna_v07_9d97e38c114bd0a2_015,536.7917,0.5127,0.7841,0.5546,1329,6238,0.1647,7.1914


Parameter terbaik:
{
  "n_estimators": 1200,
  "max_depth": 2,
  "learning_rate": 0.033437229781240425,
  "min_child_weight": 3.267956684119835,
  "reg_lambda": 1.1443495152247942,
  "reg_alpha": 3.69734260196448,
  "subsample": 0.7934287892256973,
  "colsample_bytree": 0.7424116408113942
}
Biaya lama/1.000: 537.7083
Biaya baru/1.000: 535.6250
Penurunan biaya relatif: +0.3874%
Trial selesai: 30
Ringkasan tersimpan: D:\proj-1\reports\experiments\xgb_optuna_v07\9d97e38c114bd0a2
